# [Demo] Evals: teste vs avaliação

Um teste dá certo ou errado. Uma avaliação mede uma taxa de sucesso sobre vários cenários, e serve pra
comparar versões do mesmo classificador antes de decidir qual sobe pra produção.

O que vamos ver:
- um golden dataset de 10 cenários, cada um com uma categoria correta inequívoca;
- uma versão que corrige erros reais da baseline, melhora a taxa geral, e ainda assim não deveria subir;
- por que decidir por regressão cenário a cenário é diferente de decidir só pela taxa geral.

## Setup

Carregamos as libs, as variáveis de ambiente (a chave `OPENAI_API_KEY` vem do `.env`) e inicializamos o
modelo uma vez, com `temperature=0`: uma avaliação que muda de resultado a cada execução não serve pra
comparar versões.

In [1]:
from typing import Literal

from pydantic import BaseModel
from langchain.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## O classificador

O mesmo classificador de intenção com saída estruturada de sempre: três categorias fixas, sem depender de
como o texto da resposta é escrito. Só existe um `classifier`; o que muda entre versões é a instrução que
passamos pra ele, não o classificador em si.

In [4]:
class IntentClassification(BaseModel):
    intent: Literal["informação", "agendamento", "urgência"]

In [5]:
classifier = model.with_structured_output(IntentClassification)

## Golden dataset

Dez mensagens de paciente, cada uma com uma categoria correta que não depende de opinião. Não são todas
óbvias: algumas testam justamente onde um classificador simples tende a confundir pressa por atendimento
com uma emergência de verdade.

In [6]:
SCENARIOS = [
    {"query": "Quero marcar uma consulta de cardiologia", "expected_intent": "agendamento"},
    {"query": "Gostaria de agendar um horário com a dermatologista", "expected_intent": "agendamento"},
    {"query": "Consigo uma consulta ainda hoje, é possível?", "expected_intent": "agendamento"},
    {"query": "Preciso ver um cardiologista essa semana, vocês têm vaga?", "expected_intent": "agendamento"},
    {"query": "Quais especialidades a clínica atende?", "expected_intent": "informação"},
    {"query": "Vocês têm atendimento aos sábados?", "expected_intent": "informação"},
    {"query": "O que eu faço em caso de emergência fora do horário de vocês?", "expected_intent": "informação"},
    {"query": "Estou com uma dor muito forte no peito, o que eu faço?", "expected_intent": "urgência"},
    {"query": "Cortei a mão trabalhando e não para de sangrar, o que eu faço?", "expected_intent": "urgência"},
    {"query": "Meu filho está com febre muito alta e teve uma convulsão agora", "expected_intent": "urgência"},
]

## Duas versões da instrução

A primeira é a instrução direta, a mesma usada no produto desde a Aula 2. A segunda tenta corrigir uma
falha real dela: separar pressa por atendimento de uma emergência de verdade.

In [7]:
PROMPT_V1 = "Classifique a mensagem do paciente da Clínica Alura em: informação, agendamento ou urgência."

In [8]:
PROMPT_V2 = (
    "Classifique a mensagem do paciente da Clínica Alura em: informação, agendamento ou urgência. "
    "Urgência é apenas quando a pessoa relata dor intensa, convulsão ou perda de consciência acontecendo "
    "agora. Se a pessoa menciona querer ser atendida ou pergunta sobre atendimento e disponibilidade de "
    "qualquer forma, classifique como agendamento."
)

## Função de avaliação

Roda o classificador contra cada cenário do golden dataset e imprime o resultado cenário a cenário.
Devolve um dicionário com o resultado de cada um, não só uma taxa: uma regressão é definida por cenário (um
caso que passava na baseline e passa a falhar), não por uma média que pode subir mesmo com uma regressão
escondida dentro dela.

In [9]:
def run_scenarios(prompt: str) -> dict[str, bool]:
    results = {}
    for case in SCENARIOS:
        result = classifier.invoke([SystemMessage(prompt), HumanMessage(case["query"])])
        ok = result.intent == case["expected_intent"]
        print(f"[{'PASS' if ok else 'FAIL'}] {case['query']}")
        results[case["query"]] = ok
    return results

`success_rate` resume o dicionário numa taxa. `find_regressions` compara dois resultados e aponta só os
cenários que passavam na baseline e pararam de passar: exatamente a definição de regressão.

In [13]:
def success_rate(results: dict[str, bool]) -> float:
    return sum(results.values()) / len(results)

In [10]:
a = [True, True, True, False]
sum(a) / len(a)

0.75

In [11]:
def find_regressions(baseline: dict[str, bool], candidate: dict[str, bool]) -> list[str]:
    return [query for query, passed in baseline.items() if passed and not candidate[query]]

## Rodando a avaliação

In [14]:
print("v1:")
results_v1 = run_scenarios(PROMPT_V1)
print(f"Taxa de sucesso: {success_rate(results_v1):.0%}\n")

print("v2:")
results_v2 = run_scenarios(PROMPT_V2)
print(f"Taxa de sucesso: {success_rate(results_v2):.0%}")

v1:
[PASS] Quero marcar uma consulta de cardiologia
[PASS] Gostaria de agendar um horário com a dermatologista
[FAIL] Consigo uma consulta ainda hoje, é possível?
[PASS] Preciso ver um cardiologista essa semana, vocês têm vaga?
[PASS] Quais especialidades a clínica atende?
[PASS] Vocês têm atendimento aos sábados?
[FAIL] O que eu faço em caso de emergência fora do horário de vocês?
[PASS] Estou com uma dor muito forte no peito, o que eu faço?
[PASS] Cortei a mão trabalhando e não para de sangrar, o que eu faço?
[PASS] Meu filho está com febre muito alta e teve uma convulsão agora
Taxa de sucesso: 80%

v2:
[PASS] Quero marcar uma consulta de cardiologia
[PASS] Gostaria de agendar um horário com a dermatologista
[PASS] Consigo uma consulta ainda hoje, é possível?
[PASS] Preciso ver um cardiologista essa semana, vocês têm vaga?
[PASS] Quais especialidades a clínica atende?
[FAIL] Vocês têm atendimento aos sábados?
[PASS] O que eu faço em caso de emergência fora do horário de vocês?
[PASS]

A v1 acerta 80%: erra os três cenários em que alguém quer ser atendido rápido ou pergunta sobre a política
de emergência, tratando isso como urgência de verdade. A v2 sobe pra 90%, corrigindo exatamente esses três.
Só que, ao ensinar "qualquer menção a atendimento ou disponibilidade é agendamento", ela derruba um caso
que a v1 acertava: "Vocês têm atendimento aos sábados?" vira agendamento em vez de informação. A taxa
melhorou, mas escondeu uma regressão real.

## Decisão de ship

In [15]:
regressions_v2 = find_regressions(results_v1, results_v2)
if regressions_v2:
    print(f"Não ship, mesmo com taxa maior: regressão em {regressions_v2}")
else:
    print("Ship: nenhuma regressão em relação a v1.")

Não ship, mesmo com taxa maior: regressão em ['Vocês têm atendimento aos sábados?']


## Diagnosticando e tentando de novo

O ganho da v2 (separar urgência real de pressa por atendimento) vale a pena manter. O problema é a regra
que ela usou pra isso: qualquer menção a "atendimento" ou "disponibilidade" virou agendamento, mesmo
quando a pergunta é sobre o horário de funcionamento da clínica, não um pedido pra ser atendido. Uma
terceira versão mantém a distinção de urgência e troca essa regra genérica por uma que separa "quero ser
atendido rápido" (agendamento) de "qual o horário de vocês" (informação).

In [16]:
PROMPT_V3 = (
    "Classifique a mensagem do paciente da Clínica Alura em: informação, agendamento ou urgência. "
    "Urgência é apenas quando a pessoa relata dor intensa, convulsão ou perda de consciência acontecendo "
    "agora. Quando a pessoa pede pra ser atendida ou marcar uma consulta rapidamente, classifique como "
    "agendamento. Perguntas sobre o horário de funcionamento da clínica, mesmo citando um dia específico, "
    "são informação, não um pedido de agendamento."
)

In [17]:
print("v3:")
results_v3 = run_scenarios(PROMPT_V3)
print(f"Taxa de sucesso: {success_rate(results_v3):.0%}")

v3:
[PASS] Quero marcar uma consulta de cardiologia
[PASS] Gostaria de agendar um horário com a dermatologista
[PASS] Consigo uma consulta ainda hoje, é possível?
[PASS] Preciso ver um cardiologista essa semana, vocês têm vaga?
[PASS] Quais especialidades a clínica atende?
[PASS] Vocês têm atendimento aos sábados?
[PASS] O que eu faço em caso de emergência fora do horário de vocês?
[PASS] Estou com uma dor muito forte no peito, o que eu faço?
[PASS] Cortei a mão trabalhando e não para de sangrar, o que eu faço?
[PASS] Meu filho está com febre muito alta e teve uma convulsão agora
Taxa de sucesso: 100%


A v3 acerta os dez cenários: mantém a distinção entre pressa por atendimento e emergência de verdade que a
v2 tinha acertado, e volta a classificar "vocês têm atendimento aos sábados?" como informação. Nenhuma
regressão em relação à v1, e os três acertos que a v2 tinha trazido continuam lá.

## A régua não muda

In [18]:
regressions_v3 = find_regressions(results_v1, results_v3)
if regressions_v3:
    print(f"Não ship: regressão em {regressions_v3}")
else:
    print("Ship: nenhuma regressão em relação a v1.")

Ship: nenhuma regressão em relação a v1.


## Takeaway

A taxa geral pode subir e ainda esconder uma regressão: uma versão pode corrigir vários erros reais da
baseline e, no meio da correção, quebrar um cenário que funcionava. O que decide o ship é regressão por
cenário, comparado com a baseline, nunca a taxa sozinha. Quando aparece uma regressão, o próximo passo é
diagnosticar o motivo e testar de novo.